In [2]:
import pandas as pd
import sys
import os
import numpy as np
from tqdm.auto import tqdm
tqdm.pandas()

sys.path.append(os.path.abspath("../.."))
from optimizer import variables

# sys.path.insert(0, os.path.abspath("../../../Generic-Parallel-Compute-Helper/")); from parallel_compute import *


pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [3]:
historical_price_df = pd.read_csv('2_Processed_data/historical_price.csv')
historical_generation_df = pd.read_csv('2_Processed_data/historical_generation.csv')
future_price_df = pd.read_csv('2_Processed_data/future_price.csv')

In [4]:
def run_monte_carlo_sampling_type_1(df, variable):
    n_samples = 1000
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    df[variable] = pd.to_numeric(df[variable], errors='coerce').round(2)
    years = sorted(df['Date'].dt.year.unique())
    target_hours = 8760
    
    np.random.seed(42)
    
    sample_years = np.zeros((target_hours, n_samples))
    
    for sim_num in tqdm(range(n_samples), desc="Simulations"):
        year_arr = []
        # for each 1 sample, iterate 12 times
        for month in range(1, 13):
            # get a random year
            random_year = np.random.choice(years)
            
            # Get the matching month and the random year
            month_arr = df[
                (df['Date'].dt.year == random_year) &
                (df['Date'].dt.month == month)
            ].sort_values('Date')[variable].to_numpy()
            ### This will be a month of values at a time
            
            # Add the month of values to 'year_arr'
            year_arr.extend(month_arr)
        
        
        year_arr = np.array(year_arr)
        
        if len(year_arr) > target_hours:
            year_arr = year_arr[:target_hours]
        # Store each sampled year as a column
        sample_years[:, sim_num] = year_arr
    
    # Clip negative generation, keep negative prices
    if variable == "Generation":
        sample_years = np.clip(sample_years, a_min=0, a_max=None)

    sample_years_df = pd.DataFrame(sample_years, columns=[f"Year {i + 1}" for i in range(n_samples)])
    sample_years_df.to_csv(f"2_Processed_data/resampled_{variable.lower()}.csv", index=False)


def _run_sampling_task(task):
    return run_monte_carlo_sampling_type_1(df=task["df"], variable=task["variable"])


tasks = [
    {"df": historical_price_df, "variable": "Price"},
    {"df": historical_generation_df, "variable": "Generation"},
]

# results = run_parallel(
#     function=_run_sampling_task,
#     data=tasks
# )


In [5]:
def run_monte_carlo_sampling_type_2():
    
    def load_sampling_superset(df=future_price_df):
        superset = df.copy()
        superset['date'] = pd.to_datetime(superset['Date'])
        superset["price"] = pd.to_numeric(superset["Price"], errors='coerce').round(2)
        return superset

    superset = load_sampling_superset()
    number_of_samples_to_perform = 1000
    np.random.seed(42)

    # Build the exact target timeline: one year before operation start through operation end.
    start_date = pd.to_datetime(variables.operation_start_date, dayfirst=True) - pd.DateOffset(years=1)
    end_date = pd.to_datetime(variables.operation_end_date, dayfirst=True)
    freq = f"{variables.operation_granularity_in_minutes}min"
    target_date_index = pd.date_range(start=start_date, end=end_date, freq=freq)
    target_periods = len(target_date_index)

    # Pre-build lookup dict once to avoid repeated DataFrame filtering in inner loop.
    available_years = superset.date.dt.year.unique()
    price_lookup = {
        (int(year), int(month), int(hour)): group['price'].values
        for (year, month, hour), group in superset.groupby(
            [superset.date.dt.year, superset.date.dt.month, superset.date.dt.hour]
        )
    }

    samples = []
    for sample_number in tqdm(range(number_of_samples_to_perform)):
        sample_data = []

        # Keep generating months until we have exactly the required number of points.
        while len(sample_data) < target_periods:
            for right_cycle_month in range(1, 13):
                random_year = int(np.random.choice(available_years))
                days_in_month = pd.Timestamp(year=random_year, month=right_cycle_month, day=1).days_in_month

                # Sample all days for each hour at once, then interleave into chronological order.
                month_data = np.array([
                    np.random.choice(price_lookup[(random_year, right_cycle_month, h)], size=days_in_month)
                    for h in range(24)
                ]).T.flatten()  # shape (days_in_month, 24) -> row-major = chronological

                remaining = target_periods - len(sample_data)
                sample_data.extend(month_data[:remaining])

                if len(sample_data) >= target_periods:
                    break

        samples.append(sample_data)

    samples = pd.DataFrame(np.array(samples).T, columns=[f"Sample {i + 1}" for i in range(number_of_samples_to_perform)])
    samples['Date'] = target_date_index
    samples.to_csv("2_Processed_data/resampled_aurora_price.csv", index=False)
    return samples

samples = run_monte_carlo_sampling_type_2()
display(samples[:10])


  0%|          | 0/1000 [00:00<?, ?it/s]

Sample 1  Sample 2  Sample 3  Sample 4  Sample 5  Sample 6  Sample 7  \
0    116.74    150.05    126.38    153.82    154.94    117.08    205.42   
1     82.68    132.72    108.76    158.95     62.54     60.50    103.20   
2     72.56     86.25    220.48     36.16    142.45     90.12    136.20   
3     83.54    106.05    115.10    104.21    130.01     67.76     99.52   
4    117.24    175.80    163.17    113.98    113.70     77.35    156.97   
5    113.26    113.36    159.92    155.04    147.15     69.12      1.46   
6     40.90    103.19     62.48     55.32     27.16     70.33     57.36   
7     23.42     70.96     77.59     -4.70    141.28    115.64    202.52   
8     16.45     -2.47      0.77      1.94    185.34     32.19     77.69   
9     65.19    163.26     85.19      9.88     74.25     55.32     24.78   

   Sample 8  Sample 9  Sample 10  Sample 11  Sample 12  Sample 13  Sample 14  \
0    119.26     32.09     119.94     137.80     176.48     109.97      64.76   
1    130.55    123.76     129.67      50.44     134.90     148.30      79.87   
2    121.93    148.40     120.09      82.62     136.88      25.22     112.78   
3    129.52     56.58      46.52     167.67     135.33     111.28     121.64   
4    141.00    154.22      78.94      68.58     156.68     110.36      97.48   
5    116.84    150.58     125.80     101.89     117.32     113.12      87.32   
6     27.16     72.22      73.91     165.44     110.22      88.28     241.68   
7     83.01     -1.50      94.04      33.25     106.00     100.58      24.16   
8     36.69    109.29     109.10      -0.58      62.34     104.74      48.74   
9      0.58    125.02      79.47      21.16      15.10       0.54      28.08   

   Sample 15  Sample 16  Sample 17  Sample 18  Sample 19  Sample 20  \
0     152.71     129.57     163.26     214.47     135.29      74.74   
1      94.38     125.36      61.38      65.97     110.12      71.59   
2     223.09      34.61     133.01     188.38     132.14     192.98   
3      80.83     159.92      66.99     113.94      16.12     163.41   
4      88.82     141.00      56.82     177.83     125.17     106.82   
5     225.90     160.06     230.74     149.46     151.21      42.69   
6      92.79      71.63      73.72     133.20      -9.97      44.44   
7      -3.63     101.55      81.08      23.38     143.47      -2.76   
8     135.72      70.86      44.10      34.32     123.67      35.24   
9       3.39      85.00      -0.63     178.84     150.73     156.68   

   Sample 21  Sample 22  Sample 23  Sample 24  Sample 25  Sample 26  \
0      66.99     156.92     548.74     116.46     161.38      23.52   
1      82.48     133.06      99.90     164.42     136.39     164.33   
2      56.92     162.92     160.70     159.14     132.43     151.40   
3      86.64     153.10     150.05     113.75     151.12     175.85   
4     407.22     176.04     132.04     148.94      68.34     110.74   
5      61.28     188.48     160.06     159.44     107.21      32.58   
6      91.38      63.60      57.06      80.74      74.10     117.28   
7      47.53      94.87      21.78      17.38      50.92      77.59   
8      50.34       3.92       1.11       2.46      70.24      58.76   
9      79.82      19.60       0.54      94.19     103.14     159.54   

   Sample 27  Sample 28  Sample 29  Sample 30  Sample 31  Sample 32  \
0      86.50      69.12      54.21     136.69      69.12      87.32   
1     117.48      62.15      84.51     102.42      27.50      94.92   
2     117.42      70.09     121.20     132.43      89.69      87.46   
3     115.64      79.18      75.02     151.12      98.64      72.80   
4     110.46     183.10     120.14     114.18     154.60      57.69   
5     111.23      39.54      84.12     224.59      37.08      44.87   
6     126.14     108.38      54.74      27.93      55.32      56.63   
7      37.61      13.55      29.00       0.68     157.46      95.11   
8      77.54      89.79      16.60       1.60       1.94      90.46   
9      66.46      60.8

In [6]:
# display(samples[-10:])

In [7]:
def create_bess_designs(
    duration_low_h,
    duration_high_h,
    duration_step_h,
    power_low_mw,
    power_high_mw,
    power_step_mw,
    interval_minutes,
    usable_fraction,
    output_csv_path="2_Processed_data/bess_designs.csv",
    round_dp=4,
):

    def _inclusive_range(low, high, step, ndigits):
        values = []
        current = float(low)
        eps = 10 ** (-(ndigits + 2))
        while current <= float(high) + eps:
            values.append(round(current, ndigits))
            current += float(step)
        return np.array(values, dtype=float)

    durations = _inclusive_range(duration_low_h, duration_high_h, duration_step_h, round_dp)
    powers = _inclusive_range(power_low_mw, power_high_mw, power_step_mw, round_dp)

    rows = []
    interval_hours = interval_minutes / 60.0

    for duration_h in durations:
        for power_mw in powers:
            bess_mwh = power_mw * duration_h * usable_fraction
            bess_mwh_per_interval = power_mw * interval_hours

            rows.append({
                "BESS_Duration_h": round(float(duration_h), round_dp),
                "BESS_Power_MW": round(float(power_mw), round_dp),
                "BESS_MWh": round(float(bess_mwh), round_dp),
                "BESS_MWh_per_interval": round(float(bess_mwh_per_interval), round_dp),
            })

    bess_designs_df = pd.DataFrame(rows).drop_duplicates().sort_values(
        ["BESS_Duration_h", "BESS_Power_MW"]
    ).reset_index(drop=True)

    bess_designs_df.to_csv(output_csv_path, index=False)

    return bess_designs_df


# bess_designs_df = create_bess_designs(
#     duration_low_h=2,
#     duration_high_h=8.0,
#     duration_step_h=0.5,
#     power_low_mw=4.96,
#     power_high_mw=4.96,
#     power_step_mw=4.96,
#     interval_minutes=variables.operation_granularity_in_minutes,
#     usable_fraction=variables.bess_hours_usable_fraction,
#     output_csv_path="2_Processed_data/bess_designs.csv",
# )

# bess_designs_df

In [8]:
def create_scaled_generation_sets(
    power_low_mw,        
    power_high_mw,         
    power_step_mw,        
    input_csv_path,        
    output_csv_path,      
    round_dp=4,           
):

    # Create an inclusive float range helper (includes high bound when aligned).
    def _inclusive_range(low, high, step, ndigits):
        values = []                        # Collect generated values.
        current = float(low)               # Start at lower bound.
        eps = 10 ** (-(ndigits + 2))       # Tiny tolerance for float comparisons.
        while current <= float(high) + eps:
            values.append(round(current, ndigits))  # Store rounded value.
            current += float(step)         # Move to next value by step.
        return np.array(values, dtype=float)  # Return as numpy array for iteration.

    generation_df = pd.read_csv(input_csv_path)
    base_generation = pd.to_numeric(generation_df["Generation"], errors="coerce")
    base_generation = np.clip(base_generation.to_numpy(dtype=float), a_min=0.0, a_max=None)
    max_generation = float(np.max(base_generation)) if len(base_generation) else 0.0
    normalized_profile = np.clip(base_generation / max_generation, a_min=0.0, a_max=1.0)
    power_values = _inclusive_range(power_low_mw, power_high_mw, power_step_mw, round_dp)

    scaled = {}
    for power_mw in power_values:
        col_name = f"{power_mw:g}"             
        scaled[col_name] = np.round(normalized_profile * float(power_mw), round_dp)  

    scaled_df = pd.DataFrame(scaled)
    scaled_df.to_csv(output_csv_path, index=False)
    return scaled_df


# scaled_generation_df = create_scaled_generation_sets(
#     power_low_mw=0.0,
#     power_high_mw=7.0,
#     power_step_mw=1.0,
#     input_csv_path="2_Processed_data/generation.csv",
#     output_csv_path="2_Processed_data/resampled_generation_scaled.csv",
# )

# scaled_generation_df[:100]